In [39]:
import os

ROOT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT_DIR, "data")
MODEL_DIR = os.path.join(ROOT_DIR, "models")

RAW_DIR = os.path.join(DATA_DIR, "raw")
IMAGE_DIR = os.path.join(RAW_DIR, "images", "images")
ANNOT_DIR = os.path.join(RAW_DIR, "annotations", "annotations")

BBOX_DIR = os.path.join(ANNOT_DIR, "xmls")
TRAINVAL_LABEL_PATH = os.path.join(ANNOT_DIR, "trainval.txt")
TEST_LABEL_PATH = os.path.join(ANNOT_DIR, "test.txt")

In [40]:
# 딕셔너리 반환 함수 생성
def get_label_dict(txt_file):
    global info_list

    f = open(txt_file, "r")
    lines = f.readlines()

    # \n 제거
    info_list = [line.strip() for line in lines]

    label_dict = {
        item.split(" ")[0]: {
            "class_id": item.split(" ")[1],
            "species": item.split(" ")[2],
            "breed_id": item.split(" ")[3]
        }
        for item in info_list
    }

    return label_dict


trainval_label_dict = get_label_dict(TRAINVAL_LABEL_PATH)
test_label_dict = get_label_dict(TEST_LABEL_PATH)

print(f"[레이블 개수]")
print(f"· 총 {len(trainval_label_dict) + len(test_label_dict)}개")
print(f"· Train/Val : {len(trainval_label_dict)}개")
print(f"· Test : {len(test_label_dict)}개")

print(f"\n[레이블 샘플 출력]")
print(f"· {trainval_label_dict}")

[레이블 개수]
· 총 7349개
· Train/Val : 3680개
· Test : 3669개

[레이블 샘플 출력]
· {'Abyssinian_100': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_101': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_102': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_103': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_104': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_105': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_106': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_107': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_108': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_109': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_10': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_110': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_111': {'class_id': '1', 'species': '1', 'breed_id': '1'}, 'Abyssinian_112': {'c

In [41]:
trainval_label_set = set(trainval_label_dict.keys())
test_label_set = set(test_label_dict.keys())

all_image_set = {file_name.split(".")[0].strip() for file_name in os.listdir(IMAGE_DIR)}
all_bbox_set = {file_name.split(".")[0].strip() for file_name in os.listdir(BBOX_DIR)}

trainval_image_set = all_image_set & all_bbox_set & trainval_label_set
test_image_set = all_image_set & all_bbox_set & test_label_set

trainval_bbox_set = trainval_image_set & all_bbox_set & trainval_label_set
test_bbox_set = test_image_set & (all_bbox_set - trainval_bbox_set) & test_label_set

In [42]:
from glob import glob
import xml.etree.ElementTree as ET

bbox_path_list = sorted(glob(os.path.join(BBOX_DIR, "*.xml")))

multi_object_name_list = []

for path in bbox_path_list:
    with open(path) as f:
        tree = ET.parse(f)
        root = tree.getroot()
    
    objects = root.findall('object')
    num_objects = len(objects)
    
    if num_objects > 1:
        multi_object_name_list.append(path.split('/')[-1].split(".")[0])

In [43]:
trainval_name_list = sorted(list(trainval_image_set & trainval_label_set & trainval_bbox_set))
del_name_list = list(trainval_label_set - trainval_image_set - trainval_bbox_set)

trainval_name_list = [
    name for name in trainval_name_list 
    if name not in (del_name_list and multi_object_name_list)
]

TRAINVAL_IMAGE_LIST = [f"{os.path.join(IMAGE_DIR, name)}.jpg" for name in trainval_name_list]
TRAINVAL_BBOX_LIST = [f"{os.path.join(BBOX_DIR, name)}.xml" for name in trainval_name_list]
TRAINVAL_LABEL_LIST = [int(trainval_label_dict[name]["species"]) for name in trainval_name_list]

In [44]:
print(f"[클래스별 개수]")

total_count = len(TRAINVAL_LABEL_LIST)

dog_count = sum(TRAINVAL_LABEL_LIST) - total_count
cat_count = total_count - dog_count

NUM_CLASSES = 3 # 고양이/개/배경

print(f"· 총 {total_count}개")
print(f"· cat: {cat_count}개 ({cat_count/total_count*100:.2f}%)")
print(f"· dog: {dog_count}개 ({dog_count/total_count*100:.2f}%)")

[클래스별 개수]
· 총 3670개
· cat: 1180개 (32.15%)
· dog: 2490개 (67.85%)


In [45]:
# bbox 추출 함수 생성
def get_bbox(img_path):
    with open(img_path) as f:
        tree = ET.parse(f)
        root = tree.getroot()
    
    for obj in root.findall("object"):      
        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)
            
    return [xmin, ymin, xmax, ymax]

In [46]:
import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2
from PIL import Image


class CatDogDataset(Dataset):
    def __init__(self):
        self.images = TRAINVAL_IMAGE_LIST
        self.bboxes = TRAINVAL_BBOX_LIST
        self.labels = TRAINVAL_LABEL_LIST
        self.transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(dtype=torch.float32, scale=True)
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.open(self.images[index]).convert("RGB")
        box = get_bbox(self.bboxes[index])
        label = self.labels[index]

        image = self.transform(image)
        target = {
            "bbox": torch.tensor([box], dtype=torch.float32),
            "label": torch.tensor([label], dtype=torch.int64)
        }
        
        return image, target
    
    
trainval_dataset = CatDogDataset()

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

train_indices, val_indices = train_test_split(
    list(range(len(trainval_dataset))),
    test_size=0.2,
    stratify=[label for label in trainval_dataset.labels]
)

train_subset = Subset(trainval_dataset, train_indices)
val_subset = Subset(trainval_dataset, val_indices)

TRAIN_DATALOADER = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))
VAL_DATALOADER = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))

print(f"· Train: {len(train_indices)}개 ({len(train_indices)/len(trainval_dataset)*100:.2f}%)")
print(f"· Val: {len(val_indices)}개 ({len(val_indices)/len(trainval_dataset)*100:.2f}%)")

· Train: 2936개 (80.00%)
· Val: 734개 (20.00%)
